# Provincia de Santiago - Análisis de Morfologías y Red Vial por Sensor (SCAN)

Este notebook realiza el análisis espacial de intersección en el entorno de cada sensor acústico de la provincia de Santiago:
- **Morfología de parches Liu:** Intersección con los parches de $1\times1$ km clasificados por GNN, expresado en área ($m^2$) y porcentaje dinámico.
- **Red vial (Calle Completa):** Intersección de la red vial agrupada y fusionada por nombre de calle dentro del círculo de análisis, con corrección de doble vía/bidireccionalidad.
- **Tramos de calle entre intersecciones:** Subdivisión de las calles del círculo en segmentos según los nodos de cruce/desvío de la red vial completa, midiendo sus longitudes.

In [12]:
# ============================================================
# 1. Importaciones y Carga de Datos
# ============================================================
import json
import math
import os
import time
import shutil
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import Point, Polygon, MultiPoint, LineString
from shapely.ops import split, unary_union

# --- VARIABLES MAESTRAS DE CONFIGURACIÓN ---
NOMBRE_CIUDAD = "santiago"   # Etiqueta de la ciudad (coincide con los pasos 0, 1 y 2)
RADIO_SCAN_M  = 150.0        # Radio de análisis del buffer en metros (ej. 150.0, 300.0)

# DATOS = Path("02_datos")
DATOS = Path("00_descarga")
MODELO_01 = Path("01_clas")
MODELO_01_IN = Path("01_clas/00_in")
APP = Path("02_ana")

DATOS_OUT = Path("02_ana")

#MODELO_DIR = Path ("01_modelo")
# LIU_DIR = Path("../scan_geo03_osm_real/liu_clasificacion")

# 1. Cargar parches clasificados (Lat/Lon) y proyectar a Web Mercator (EPSG:3857)
gdf_parches = gpd.read_file(MODELO_01 / f"parches_liu_{NOMBRE_CIUDAD.lower()}.geojson").to_crs(3857)
print(f"Parches cargados y proyectados para {NOMBRE_CIUDAD}:", len(gdf_parches))

# 2. Cargar red de calles (ya está en EPSG:3857)
edges = gpd.read_parquet(DATOS / f"red_drive_{NOMBRE_CIUDAD.lower()}.parquet")
print("Red vial cargada (aristas):", len(edges))

# 3. Cargar mediciones acústicas reales (Lat/Lon)
with open(MODELO_01 / f"mediciones_aumilab_{NOMBRE_CIUDAD.lower()}.json", "r", encoding="utf-8") as f:
    mediciones_json = json.load(f)
print("Mediciones de ruido cargadas:", len(mediciones_json))

# Convertir mediciones a GeoDataFrame y proyectar a EPSG:3857
records = []
for m in mediciones_json:
    records.append({
        "id": m["id"],
        "calle": m["calle"],
        "decibels": m["la_eq"],
        "lat": m["lat"],
        "lon": m["lon"],
        "geometry": Point(m["lon"], m["lat"])
    })
gdf_mediciones = gpd.GeoDataFrame(records, crs=4326).to_crs(3857)
print("Mediciones proyectadas a EPSG:3857.")

# Cargar clases y colores de Liu
with open(MODELO_01_IN / "clases_liu.json", encoding="utf-8") as f:
    meta_liu = json.load(f)
CLASES = {int(k): v for k, v in meta_liu["clases"].items()}
COLORES_MRF = {v["nombre"]: v["color"] for v in CLASES.values()}


Parches cargados y proyectados para santiago: 10811
Red vial cargada (aristas): 238238
Mediciones de ruido cargadas: 34
Mediciones proyectadas a EPSG:3857.


In [13]:
# ============================================================
# 2. Detección de Nodos de Intersección de la Red Vial
# ============================================================
print("Extrayendo nodos únicos de la red vial...")
t0 = time.time()
coords = set()
for g in edges.geometry.values:
    if g.coords:
        coords.add(g.coords[0])
        coords.add(g.coords[-1])
print("Nodos viales únicos encontrados:", len(coords))
unique_points = [Point(xy) for xy in coords]
gdf_nodos = gpd.GeoDataFrame(geometry=unique_points, crs=3857)
sindex_nodos = gdf_nodos.sindex
print(f"Indexados espacialmente en {round(time.time() - t0, 2)} s")


Extrayendo nodos únicos de la red vial...
Nodos viales únicos encontrados: 97072
Indexados espacialmente en 2.7 s


In [14]:
# ============================================================
# 3. Definición del Radio, Configuración de Carpetas e Indexación
# ============================================================
radio_efectivo = RADIO_SCAN_M  # Si es None, el radio por defecto es 150 metros. Puedes definir un número aquí.

# Determinar el radio efectivo para nombrar la carpeta
# radio_efectivo = RADIO_SCAN if RADIO_SCAN is not None else 150.0

# Crear carpeta de salidas con marca de tiempo única de esta corrida
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(DATOS_OUT / f"salidas_scan_{NOMBRE_CIUDAD.lower()}_r{int(radio_efectivo)}_{timestamp}")
OUTPUT_DIR.mkdir(exist_ok=True)

# Revisarlas...
FIGS_DIR = OUTPUT_DIR / "figuras_paso_a_paso"
FIGS_DIR.mkdir(exist_ok=True)

print(f"Resultados de esta prueba se guardarán en: {OUTPUT_DIR}")

sindex_parches = gdf_parches.sindex
sindex_edges = edges.sindex

# Tipos de via presentes en toda la red (para columnas ML consistentes entre scans)
TIPOS_CALLE = sorted({str(t).lower() for t in edges["highway_osm"].dropna().unique()} | {"local"})

def sanitizar(text):
    if not text:
        return "desconocido"
    return str(text).lower().replace(" ", "_").replace("&", "y").replace("-", "_")

def geometry_to_coordinates(geom):
    if geom.is_empty:
        return []
    gs = gpd.GeoSeries([geom], crs=3857).to_crs(4326).iloc[0]
    if gs.geom_type == 'LineString':
        return [{"lat": round(y, 7), "lon": round(x, 7)} for x, y in gs.coords]
    elif gs.geom_type == 'MultiLineString':
        coords = []
        for line in gs.geoms:
            coords.extend([{"lat": round(y, 7), "lon": round(x, 7)} for x, y in line.coords])
        return coords
    return []

def get_coords_3857_list(geom):
    if geom.is_empty:
        return []
    if geom.geom_type == 'LineString':
        return [[(pt[0], pt[1]) for pt in geom.coords]]
    elif geom.geom_type == 'MultiLineString':
        return [[(pt[0], pt[1]) for pt in line.coords] for line in geom.geoms]
    return []

def ejecutar_scan(med_row):
    measurement_id = med_row["id"]
    centro_geom = med_row["geometry"]
    
    # radio = RADIO_SCAN if RADIO_SCAN is not None else 150.0
    radio = RADIO_SCAN_M
    circulo = centro_geom.buffer(radio, resolution=256)
    area_circulo = circulo.area

    # 1. -------- Parches Liu ∩ SCN --------
    idxs_parches = sindex_parches.query(circulo, predicate="intersects")
    resultados_parches_liu = []
    
    for idx in idxs_parches:
        parche = gdf_parches.iloc[idx]
        inter = parche.geometry.intersection(circulo)
        if inter.area > 0:
            porcentaje = (inter.area / parche.geometry.area) * 100.0
            resultados_parches_liu.append({
                "parche_id": parche["id"],
                "liu_class": parche["liu_class"],
                "area_en_circulo_m2": round(inter.area, 4),
                "porcentaje_en_circulo": round(porcentaje, 4)
            })

    # 2. -------- Calles (Red Vial) ∩ SCN (Deduplicadas por Doble Vía / Sentido) --------
    idxs_edges = sindex_edges.query(circulo, predicate="intersects")
    tramos_por_nombre = {}
    tipos_por_nombre = {}

    # Deduplicar aristas gemelas de doble sentido (u->v / v->u con la misma key)
    seen_calles = set()
    for idx in idxs_edges:
        calle = edges.iloc[idx]
        seg_id = (min(calle["u"], calle["v"]), max(calle["u"], calle["v"]), calle["key"])
        if seg_id in seen_calles:
            continue
        seen_calles.add(seg_id)
        tramo = calle.geometry.intersection(circulo)
        if tramo.length > 0:
            nombre_calle = calle["nombre"] if pd.notna(calle["nombre"]) else "s/n"
            tipo_calle = calle["highway_osm"] if pd.notna(calle["highway_osm"]) else "local"

            if nombre_calle not in tramos_por_nombre:
                tramos_por_nombre[nombre_calle] = []
                tipos_por_nombre[nombre_calle] = tipo_calle
            tramos_por_nombre[nombre_calle].append(tramo)

    resultados_calles = []
    str_idx = 1
    largo_calles_total = 0.0

    for nombre, lista_tramos in tramos_por_nombre.items():
        geom_fusionada = unary_union(lista_tramos)
        total_len = sum(t.length for t in lista_tramos)
        largo_calles_total += total_len
        
        resultados_calles.append({
            "str_id": f"SC_{str_idx}",
            "str_name": nombre,
            "str_type": tipos_por_nombre[nombre],
            "length_en_circulo_m": round(total_len, 4),
            "coordinates": geometry_to_coordinates(geom_fusionada),
            "coordinates_3857_list": get_coords_3857_list(geom_fusionada)
        })
        str_idx += 1

    # 3. -------- Tramos de calle entre intersecciones --------
    idxs_nodos = sindex_nodos.query(circulo, predicate="intersects")
    nodos_en_circulo = MultiPoint([unique_points[i] for i in idxs_nodos]) if len(idxs_nodos) > 0 else MultiPoint()
    
    resultados_tramos_interseccion = []
    tramo_idx = 1
    seen_segments = set()
    
    for idx in idxs_edges:
        calle = edges.iloc[idx]
        
        # Deduplicar segmentos bidireccionales de línea única
        u, v, k = calle["u"], calle["v"], calle["key"]
        seg_id = (min(u, v), max(u, v), k)
        if seg_id in seen_segments:
            continue
        seen_segments.add(seg_id)
        
        tramo = calle.geometry.intersection(circulo)
        if tramo.length > 0:
            nombre_calle = calle["nombre"] if pd.notna(calle["nombre"]) else "s/n"
            type_calle = calle["highway_osm"] if pd.notna(calle["highway_osm"]) else "local"
            
            if not nodos_en_circulo.is_empty:
                split_result = split(tramo, nodos_en_circulo)
                if split_result.geom_type == 'GeometryCollection':
                    geoms = list(split_result.geoms)
                else:
                    geoms = [split_result]
            else:
                geoms = [tramo]
                
            for g in geoms:
                if g.geom_type == 'LineString' and g.length > 0.1:
                    resultados_tramos_interseccion.append({
                        "section_id": tramo_idx,
                        "str_id": str(idx),
                        "str_name": nombre_calle,
                        "str_type": type_calle,
                        "length_m": round(g.length, 4),
                        "coordinates": geometry_to_coordinates(g),
                        "coordinates_3857": [(pt[0], pt[1]) for pt in g.coords]
                    })
                    tramo_idx += 1

    # 4. -------- ML features --------
    ml_features = {}
    
    clases_liu_todas = ["Regular Grid", "Irregular Grid", "Broken Grid", "Warped Parallel", "Loops & Lollipops", "Sparse"]
    for clase in clases_liu_todas:
        tipo = sanitizar(clase)
        ml_features[f"area_parche_{tipo}_m2"] = 0.0
        ml_features[f"porcentaje_parche_{tipo}"] = 0.0

    # Cada parche aporta el % relativo a su propia area (suma de porcentaje_en_circulo)
    for r in resultados_parches_liu:
        tipo = sanitizar(r["liu_class"])
        key_area = f"area_parche_{tipo}_m2"
        key_pct = f"porcentaje_parche_{tipo}"
        ml_features[key_area] = round(ml_features.get(key_area, 0.0) + r["area_en_circulo_m2"], 4)
        ml_features[key_pct] = round(ml_features.get(key_pct, 0.0) + r["porcentaje_en_circulo"], 4)

    ml_features["largo_calles_total_m"] = round(largo_calles_total, 4)
    ml_features["cantidad_calles"] = len(resultados_calles)
    ml_features["densidad_vial_m_por_m2"] = round(ml_features["largo_calles_total_m"] / area_circulo, 8)
    
    for tipo in TIPOS_CALLE:
        ml_features[f"largo_calles_{tipo}_m"] = 0.0

    for r in resultados_calles:
        tipo = r["str_type"].lower()
        key = f"largo_calles_{tipo}_m"
        ml_features[key] = round(ml_features.get(key, 0) + r["length_en_circulo_m"], 4)

    ml_features["cantidad_tramos_interseccion"] = len(resultados_tramos_interseccion)
    ml_features["largo_promedio_tramo_interseccion_m"] = round(
        sum(t["length_m"] for t in resultados_tramos_interseccion) / len(resultados_tramos_interseccion), 4
    ) if resultados_tramos_interseccion else 0.0

    scn_result = {
        "scan_id": str(measurement_id),
        "noise_id": str(measurement_id),
        "noise_measurement": {
            "id": str(med_row["id"]),
            "calle": med_row["calle"],
            "lat": med_row["lat"],
            "lon": med_row["lon"],
            "decibels": med_row["decibels"]
        },
        "scan": {
            "id": str(measurement_id),
            "center": {"lat": med_row["lat"], "lon": med_row["lon"]},
            "radius_mts": radio
        },
        "morfologia_parches_liu": resultados_parches_liu,
        "calles": resultados_calles,
        "tramos_interseccion": resultados_tramos_interseccion,
        "ml_features": ml_features
    }

    ml_dataset_row = {
        "scan_id": str(measurement_id),
        "noise_id": str(measurement_id),
        "lat": med_row["lat"],
        "lon": med_row["lon"],
        "decibels": med_row["decibels"],
        "scan_radius_mts": radio,
        **ml_features
    }

    return scn_result, ml_dataset_row, circulo


Resultados de esta prueba se guardarán en: 02_ana/salidas_scan_santiago_r150_20260916_204530


In [15]:
# ============================================================
# 4. Funciones de Visualización
# ============================================================
def dibujar_base_local(ax, circulo):
    caja_ampliada = circulo.buffer(250.0)
    
    # 1. Graficar parches de fondo
    idxs_p = sindex_parches.query(caja_ampliada, predicate="intersects")
    for idx in idxs_p:
        p = gdf_parches.iloc[idx]
        x, y = p.geometry.exterior.xy
        ax.fill(x, y, color='#f5f5f7', edgecolor='#e1e1e6', alpha=0.9, zorder=1)
        
    # 2. Graficar calles de fondo
    idxs_e = sindex_edges.query(caja_ampliada, predicate="intersects")
    for idx in idxs_e:
        e = edges.iloc[idx]
        x, y = e.geometry.xy
        ax.plot(x, y, color='#e4e4eb', linewidth=1.5, zorder=2)

def zoom_a_scan(ax, circulo):
    c_x, c_y = circulo.centroid.x, circulo.centroid.y
    r = circulo.bounds[2] - c_x
    ax.set_xlim(c_x - r - 50, c_x + r + 50)
    ax.set_ylim(c_y - r - 50, c_y + r + 50)
    ax.set_aspect('equal')

def visualizar_base_scan(scn_result, circulo):
    scan_id = scn_result["scan_id"]
    fig, ax = plt.subplots(figsize=(8, 8))
    
    dibujar_base_local(ax, circulo)
    
    # Círculo verde del SCAN
    x, y = circulo.exterior.xy
    ax.plot(x, y, color='green', linewidth=2.5, zorder=5)

    # Mediciones y punto de centro
    meas = scn_result["noise_measurement"]
    centro_pt = Point(meas["lon"], meas["lat"])
    centro_proj = gpd.GeoSeries([centro_pt], crs=4326).to_crs(3857).iloc[0]
    
    # Color por decibelios
    db = meas["decibels"]
    color = "#c62828" if db >= 70 else ("#ef6c00" if db >= 65 else ("#f9a825" if db >= 60 else "#2e7d32"))
    
    ax.scatter([centro_proj.x], [centro_proj.y], color=color, s=220, edgecolor='white', linewidth=2.0, zorder=10)
    
    # Cuadro informativo
    ax.text(centro_proj.x, centro_proj.y + 12, f"Sensor: {meas['id']}\nCalle: {meas['calle']}\nRuido: {db} dB",
            ha='center', va='bottom', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.9, ec="grey", lw=0.8), zorder=12)

    zoom_a_scan(ax, circulo)
    ax.set_title(f"SCAN {NOMBRE_CIUDAD.lower()} {scan_id} - Punto de Medición Base", fontsize=11, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(FIGS_DIR / f'scan_{NOMBRE_CIUDAD.lower()}_{scan_id}_paso_00_base.png', dpi=150)
    plt.close(fig)

def visualizar_parches_liu_scan(scn_result, circulo):
    scan_id = scn_result["scan_id"]
    fig, ax = plt.subplots(figsize=(8, 8))
    
    dibujar_base_local(ax, circulo)
    
    # Círculo verde
    x, y = circulo.exterior.xy
    ax.plot(x, y, color='green', linewidth=2.5, zorder=5)

    for r in scn_result.get("morfologia_parches_liu", []):
        parche_orig = gdf_parches.iloc[next(i for i, pid in enumerate(gdf_parches["id"]) if pid == r["parche_id"])]
        inter = parche_orig.geometry.intersection(circulo)
        if not inter.is_empty:
            color = COLORES_MRF.get(r["liu_class"], "grey")
            if inter.geom_type == 'Polygon':
                ax.fill(*inter.exterior.xy, color=color, edgecolor='black', alpha=0.7, linewidth=1.2, linestyle='--', zorder=4)
            elif inter.geom_type == 'MultiPolygon':
                for poly in inter.geoms:
                    ax.fill(*poly.exterior.xy, color=color, edgecolor='black', alpha=0.7, linewidth=1.2, linestyle='--', zorder=4)
            
            if r["porcentaje_en_circulo"] > 0.0001:
                punto = inter.representative_point()
                ax.text(punto.x, punto.y, f'Parche {r["parche_id"]}\n{r["liu_class"]}\n{r["porcentaje_en_circulo"]:.3f}% ({r["area_en_circulo_m2"]:.0f} m2)',
                        ha='center', va='center', fontsize=8, color='black', fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.75, ec="grey", lw=0.5), zorder=6)

    zoom_a_scan(ax, circulo)
    ax.set_title(f"SCAN {NOMBRE_CIUDAD.lower()} {scan_id} - Morfologia Liu ∩ SCN", fontsize=11, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(FIGS_DIR / f'scan_{NOMBRE_CIUDAD.lower()}_{scan_id}_paso_01b_resumen_mrf_liu.png', dpi=150)
    plt.close(fig)

def visualizar_calles_scan(scn_result, circulo):
    scan_id = scn_result["scan_id"]
    fig, ax = plt.subplots(figsize=(8, 8))
    
    dibujar_base_local(ax, circulo)
    
    # Círculo verde del SCAN
    x, y = circulo.exterior.xy
    ax.plot(x, y, color='green', linewidth=2.5, zorder=5)

    colores_alternativos = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

    for idx, r in enumerate(scn_result.get("calles", [])):
        paths = r.get("coordinates_3857_list", [])
        if not paths:
            continue
        color = colores_alternativos[idx % len(colores_alternativos)]
        
        all_lines = []
        for path in paths:
            if len(path) >= 2:
                line = LineString(path)
                all_lines.append(line)
                x_line, y_line = line.xy
                ax.plot(x_line, y_line, color=color, linewidth=4.5, alpha=0.9, zorder=4)
                
        if all_lines and r["length_en_circulo_m"] > 5.0:
            longest_line = max(all_lines, key=lambda l: l.length)
            punto = longest_line.interpolate(0.5, normalized=True)
            ax.text(punto.x, punto.y + 4, f'{r["str_name"]}\n{r["length_en_circulo_m"]:.1f} m',
                    ha='center', fontsize=7.5, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.9, ec="grey", lw=0.5), zorder=6)

    zoom_a_scan(ax, circulo)
    ax.set_title(f"SCAN {NOMBRE_CIUDAD.lower()} {scan_id} - Calles Completas ∩ SCN", fontsize=11, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(FIGS_DIR / f'scan_{NOMBRE_CIUDAD.lower()}_{scan_id}_paso_02_resumen_str.png', dpi=150)
    plt.close(fig)

def visualizar_secciones_interseccion_scan(scn_result, circulo):
    scan_id = scn_result["scan_id"]
    fig, ax = plt.subplots(figsize=(8, 8))
    
    dibujar_base_local(ax, circulo)
    
    # Círculo verde
    x, y = circulo.exterior.xy
    ax.plot(x, y, color='green', linewidth=2.5, zorder=5)

    colores_alternativos = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

    for r in scn_result.get("tramos_interseccion", []):
        puntos = r.get("coordinates_3857", [])
        if len(puntos) >= 2:
            g = LineString(puntos)
            color = colores_alternativos[r["section_id"] % len(colores_alternativos)]
            x_line, y_line = g.xy
            ax.plot(x_line, y_line, color=color, linewidth=4.5, alpha=0.9, zorder=4)
            
            if r["length_m"] > 5.0:
                punto = g.interpolate(0.5, normalized=True)
                ax.text(punto.x, punto.y + 4, f'{r["str_name"]}\nT{r["section_id"]}: {r["length_m"]:.1f} m',
                        ha='center', fontsize=7.5, fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.85, ec="grey", lw=0.5), zorder=6)

    zoom_a_scan(ax, circulo)
    ax.set_title(f"SCAN {NOMBRE_CIUDAD.lower()} {scan_id} - Tramos de Calle entre Intersecciones ∩ SCN", fontsize=11, fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(FIGS_DIR / f'scan_{NOMBRE_CIUDAD.lower()}_{scan_id}_paso_03_resumen_tramos_interseccion.png', dpi=150)
    plt.close(fig)


In [16]:
# ============================================================
# 5. Ejecución del Experimento y Exportación
# ============================================================
resultados_scan = []
filas_ml = []

print("Ejecutando SCAN para todos los puntos de medición ...")
for idx, row in gdf_mediciones.iterrows():
    scn_result, ml_dataset_row, circulo = ejecutar_scan(row)
    resultados_scan.append(scn_result)
    filas_ml.append(ml_dataset_row)
    
    # Generar visualizaciones y guardarlas en el directorio único de esta corrida
    visualizar_base_scan(scn_result, circulo)
    visualizar_parches_liu_scan(scn_result, circulo)
    visualizar_calles_scan(scn_result, circulo)
    visualizar_secciones_interseccion_scan(scn_result, circulo)
    
    print(f"Sensor {row['id']} ({row['calle']}): {len(scn_result['morfologia_parches_liu'])} parches, "
          f"{len(scn_result['calles'])} calles, {len(scn_result['tramos_interseccion'])} tramos.")

# 1. Guardar en la carpeta con marca de tiempo única de esta corrida (historial)
df_ml = pd.DataFrame(filas_ml)
df_ml.to_csv(DATOS_OUT / f"ml_dataset_{NOMBRE_CIUDAD.lower()}.csv", index=False)
with open(DATOS_OUT / f"ml_dataset_{NOMBRE_CIUDAD.lower()}.json", "w", encoding="utf-8") as f:
    json.dump(filas_ml, f, ensure_ascii=False, indent=2)
with open(DATOS_OUT / f"experimento_result_{NOMBRE_CIUDAD.lower()}.json", "w", encoding="utf-8") as f:
    json.dump(resultados_scan, f, ensure_ascii=False, indent=2)


# Realizaba una copia de los archihvos generados
# # 2. Guardar/Copiar en la carpeta estándar 'salidas_scan' (sobre-escribiendo la última corrida)
# # de esta manera la App Interactiva (paso2_app_santiago.ipynb) siempre funciona con los últimos datos
# DEFAULT_OUT = Path("salidas_scan")
# DEFAULT_OUT.mkdir(exist_ok=True)
# DEFAULT_FIGS = DEFAULT_OUT / "figuras_paso_a_paso"
# DEFAULT_FIGS.mkdir(exist_ok=True)

# df_ml.to_csv(DEFAULT_OUT / f"ml_dataset_{NOMBRE_CIUDAD.lower()}.csv", index=False)
# with open(DEFAULT_OUT / f"ml_dataset_{NOMBRE_CIUDAD.lower()}.json", "w", encoding="utf-8") as f:
#     json.dump(filas_ml, f, ensure_ascii=False, indent=2)
# with open(DEFAULT_OUT / f"experimento_result_{NOMBRE_CIUDAD.lower()}.json", "w", encoding="utf-8") as f:
#     json.dump(resultados_scan, f, ensure_ascii=False, indent=2)

# # Copiar todas las figuras de la corrida al directorio por defecto
# print("Copiando figuras a la carpeta salidas_scan para actualizar la vista activa...")
# for file_name in os.listdir(FIGS_DIR):
#     full_file_name = os.path.join(FIGS_DIR, file_name)
#     if os.path.isfile(full_file_name):
#         shutil.copy(full_file_name, DEFAULT_FIGS)

print("\n--- Ejecución finalizada con éxito ---")
print("Historial de esta corrida guardado en:", OUTPUT_DIR)
#print("Resultados actualizados para la App en:", DEFAULT_OUT)


Ejecutando SCAN para todos los puntos de medición ...
Sensor C01 (La Travesía): 4 parches, 7 calles, 16 tramos.
Sensor C02 (Av. General Bonilla): 4 parches, 7 calles, 14 tramos.
Sensor C03 (Bascuñan Guerrero): 4 parches, 6 calles, 12 tramos.
Sensor C04 (San Ignacio de Loyola): 3 parches, 7 calles, 12 tramos.
Sensor C05 (Av. Domingo Santa María): 4 parches, 12 calles, 32 tramos.
Sensor C06 (Carlos Antúnez): 4 parches, 4 calles, 15 tramos.
Sensor C07 (Antonio Varas): 4 parches, 6 calles, 13 tramos.
Sensor C08 (Av. Consistorial): 3 parches, 7 calles, 29 tramos.
Sensor C09 (Av. Rojas Magallanes): 4 parches, 4 calles, 8 tramos.
Sensor C10 (Yungay): 4 parches, 10 calles, 22 tramos.
Sensor E01 (Av. Las Rejas): 4 parches, 10 calles, 38 tramos.
Sensor E02 (Av. Las Rejas (norte)): 4 parches, 13 calles, 40 tramos.
Sensor E03 (Av. Independencia): 4 parches, 7 calles, 23 tramos.
Sensor E04 (Av.Ossa): 4 parches, 10 calles, 33 tramos.
Sensor E05 (Av. Vicuña Mackenna): 4 parches, 12 calles, 31 tramos.